# EDA and Model Tuning

This notebook explores the synthetic IoT sensor stream and benchmarks the four anomaly detectors that ship with the pipeline.

**Sections**

1. Generate a labelled sample dataset
2. Visualise normal vs anomalous distributions
3. Compare detector performance with ROC curves and confusion matrices
4. Hyperparameter sweep for the Isolation Forest
5. Show why the ensemble outperforms any individual detector

In [ ]:
import sys, os
from pathlib import Path

# Allow imports from project root
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (precision_recall_fscore_support, roc_curve, auc,
                             confusion_matrix, classification_report)

from src.config import load_config
from src.data_generator import SensorDataGenerator
from models.isolation_forest import IsolationForestDetector
from models.dbscan_detector import DBSCANDetector
from models.zscore_detector import ZScoreDetector
from models.ensemble import EnsembleDetector

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_SEED = 42

## 1. Generate a labelled sample dataset

In [ ]:
cfg = load_config(ROOT / 'config' / 'config.yaml')
gen_cfg = {**cfg['data_generator']}
gen_cfg['emit_delay_seconds'] = 0.0
gen_cfg['anomaly_rate'] = 0.05

gen = SensorDataGenerator(gen_cfg, seed=RANDOM_SEED)
records = gen.generate_batch(5000)
df = pd.DataFrame(records)
print(df.shape)
df.head()

In [ ]:
df['is_true_anomaly'].value_counts(normalize=True)

In [ ]:
df[df['is_true_anomaly'] == 1]['anomaly_type'].value_counts()

## 2. Visualise normal vs anomalous distributions

In [ ]:
feature_cols = ['temperature', 'pressure', 'vibration', 'power_consumption']
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, col in zip(axes.flat, feature_cols):
    sns.histplot(data=df, x=col, hue='is_true_anomaly', bins=60,
                 stat='density', common_norm=False, ax=ax,
                 palette={0: '#1f77b4', 1: '#d62728'})
    ax.set_title(col)
fig.suptitle('Feature distributions: normal (blue) vs anomalous (red)')
fig.tight_layout()
plt.show()

In [ ]:
# Pairplot of a subsample for visual inspection of multivariate structure
sample = df.sample(min(1500, len(df)), random_state=RANDOM_SEED)
sns.pairplot(sample[feature_cols + ['is_true_anomaly']],
             hue='is_true_anomaly',
             palette={0: '#1f77b4', 1: '#d62728'},
             plot_kws=dict(alpha=0.4, s=12),
             diag_kind='kde')
plt.show()

## 3. Detector benchmark

Train each detector on the first half of the dataset (using only ground-truth normals) and score the rest.

In [ ]:
split = len(df) // 2
train_df = df.iloc[:split]
test_df  = df.iloc[split:]

train_normal = train_df[train_df['is_true_anomaly'] == 0].to_dict('records')
test_records = test_df.to_dict('records')
y_true = test_df['is_true_anomaly'].to_numpy()
print(f'Train (normal only): {len(train_normal)}  |  Test: {len(test_records)}')

In [ ]:
iforest = IsolationForestDetector(feature_cols, cfg['detectors']['isolation_forest'])
dbscan  = DBSCANDetector(feature_cols, cfg['detectors']['dbscan'])
zscore  = ZScoreDetector(feature_cols, cfg['detectors']['zscore'])
ens     = EnsembleDetector([iforest, dbscan, zscore], feature_cols, cfg['ensemble'])

for det in (iforest, dbscan, zscore):
    det.fit(train_normal)
ens.is_fitted = all(d.is_fitted for d in (iforest, dbscan, zscore))

In [ ]:
def score_detector(det, records):
    preds, scores = [], []
    for r in records:
        out = det.predict(r)
        preds.append(out.is_anomaly)
        scores.append(out.score)
    return np.array(preds), np.array(scores)

results = {}
for det in (iforest, dbscan, zscore, ens):
    p, s = score_detector(det, test_records)
    results[det.name] = (p, s)

summary = []
for name, (p, s) in results.items():
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, p, average='binary', zero_division=0)
    summary.append({'detector': name, 'precision': pr, 'recall': rc, 'f1': f1})
summary_df = pd.DataFrame(summary).set_index('detector')
summary_df.style.format('{:.3f}').background_gradient(cmap='Blues')

In [ ]:
# ROC curves (where the score is a continuous value)
fig, ax = plt.subplots(figsize=(7, 6))
for name, (p, s) in results.items():
    if np.std(s) == 0:
        continue
    fpr, tpr, _ = roc_curve(y_true, s)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.2f})')
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves')
ax.legend(loc='lower right')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, (p, _)) in zip(axes, results.items()):
    cm = confusion_matrix(y_true, p)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'], ax=ax)
    ax.set_title(name)
fig.suptitle('Confusion matrices')
fig.tight_layout()
plt.show()

## 4. Isolation Forest hyperparameter sweep

In [ ]:
rows = []
for n_est in (50, 100, 200, 300):
    for cont in (0.02, 0.04, 0.06, 0.08, 0.10):
        det = IsolationForestDetector(feature_cols, {
            'contamination': cont, 'n_estimators': n_est, 'max_samples': 256,
            'initial_train_size': 200, 'retrain_every': 99999,
        })
        det.fit(train_normal)
        preds, _ = score_detector(det, test_records)
        pr, rc, f1, _ = precision_recall_fscore_support(y_true, preds, average='binary', zero_division=0)
        rows.append({'n_estimators': n_est, 'contamination': cont,
                     'precision': pr, 'recall': rc, 'f1': f1})

sweep = pd.DataFrame(rows)
pivot = sweep.pivot(index='n_estimators', columns='contamination', values='f1')
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('Isolation Forest F1 by hyperparameters')
plt.show()

best = sweep.sort_values('f1', ascending=False).head(5)
best

## 5. Why the ensemble wins

Each detector has a different blind spot:

* **Z-Score** catches univariate spikes but misses contextual / multivariate anomalies.
* **Isolation Forest** is great at multivariate outliers but slow to react to drift.
* **DBSCAN** thrives in dense clusters but is sensitive to its `eps` parameter and ignores temporal context.

By voting, the ensemble combines complementary strengths. The cell below quantifies this by computing F1 *per anomaly type*.

In [ ]:
type_results = []
for atype in test_df['anomaly_type'].dropna().unique():
    mask = (test_df['is_true_anomaly'] == 1) & (test_df['anomaly_type'] == atype)
    idx = mask.to_numpy()
    for name, (p, _) in results.items():
        recall = (p[idx].sum() / max(idx.sum(), 1))
        type_results.append({'detector': name, 'anomaly_type': atype, 'recall': recall})

type_df = pd.DataFrame(type_results)
pivot = type_df.pivot(index='detector', columns='anomaly_type', values='recall')
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1)
plt.title('Recall by anomaly type')
plt.show()

### Takeaways

* The ensemble's recall on each anomaly type is at least as good as the best individual detector.
* For point anomalies, Z-Score is enough.
* For multivariate and drift anomalies, Isolation Forest dominates.
* DBSCAN catches contextual anomalies that other detectors miss.
* The weighted vote in `EnsembleDetector` is configurable in `config/config.yaml` — bumping the `weight` of an under-performing detector smoothly trades off precision against recall.